# **DX 799: Week 2 — Linear Regression 2**

For Week 2, include concepts such as linear regression with lasso, ridge, and elastic net regression. This homework will be submitted for peer review and feedback in Week 3 in the assignment titled 3.4 Peer Review: Week 2 Jupyter Notebook. Complete your Jupyter Notebook homework by 11:59 pm ET on Sunday.

In [22]:
import numpy as np
import pandas as pd
import scipy
import statsmodels.api as sm
import seaborn as sns
from statsmodels.stats.outliers_influence import variance_inflation_factor
import statsmodels.formula.api as smf
from sklearn.model_selection import KFold
from sklearn.metrics import root_mean_squared_error
import matplotlib.pyplot as plt
from sklearn.model_selection import RepeatedKFold, cross_val_score
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import BaggingRegressor, RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import RepeatedKFold, cross_val_score

pd.set_option('display.max_columns', None)

---

# DIABETES DATASET

In [23]:
#DIABETES: LOAD
df_diabetes = pd.read_csv("../Datasets/diabetes_cleaned.csv")

df_diabetes.iloc[0:5]

,Diabetes_012,HighBP,HighChol,CholCheck,BMI,Smoker,Stroke,HeartDiseaseorAttack,PhysActivity,Fruits,Veggies,HvyAlcoholConsump,AnyHealthcare,NoDocbcCost,GenHlth,MentHlth,PhysHlth,DiffWalk,Sex,Age,Education,Income
0,0.0,1.0,1.0,1.0,40.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,5.0,18.0,15.0,1.0,0.0,9.0,4.0,3.0
1,0.0,0.0,0.0,0.0,25.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,3.0,0.0,0.0,0.0,0.0,7.0,6.0,1.0
2,0.0,1.0,1.0,1.0,28.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,1.0,5.0,30.0,30.0,1.0,0.0,9.0,4.0,8.0
3,0.0,1.0,0.0,1.0,27.0,0.0,0.0,0.0,1.0,1.0,1.0,0.0,1.0,0.0,2.0,0.0,0.0,0.0,0.0,11.0,3.0,6.0
4,0.0,1.0,1.0,1.0,24.0,0.0,0.0,0.0,1.0,1.0,1.0,0.0,1.0,0.0,2.0,3.0,0.0,0.0,0.0,11.0,5.0,4.0


**Diabetes: X, y, scale, CV**

In [24]:
y_diabetes = df_diabetes["BMI"]
X_diabetes = df_diabetes.drop("BMI", axis=1)

cv_diabetes = RepeatedKFold(n_splits=5, n_repeats=5, random_state=42)

**Diabetes: Ridge Regression**

In [25]:
ridge_pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("model", Ridge(alpha=1.0, random_state=42))
])

ridge_scores = -cross_val_score(ridge_pipe, X_diabetes, y_diabetes,
                                scoring="neg_mean_absolute_error",
                                cv=cv_diabetes)
print(f"Diabetes Ridge - Mean MAE: {np.mean(ridge_scores):.4f}, Std: {np.std(ridge_scores):.4f}")

Diabetes Ridge - Mean MAE: 4.3228, Std: 0.0155


**Diabetes: Lasso Regression** 

In [26]:
lasso_pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("model", Lasso(alpha=0.01, random_state=42, max_iter=10000))
])
lasso_scores = -cross_val_score(lasso_pipe, X_diabetes, y_diabetes,
                                scoring="neg_mean_absolute_error", cv=cv_diabetes)
print(f"Diabetes Lasso - Mean MAE: {np.mean(lasso_scores):.4f}, Std: {np.std(lasso_scores):.4f}")


Diabetes Lasso - Mean MAE: 4.3226, Std: 0.0156


**Diabetes: Elastic Regression**

In [27]:
elastic_pipe = Pipeline([("scaler", StandardScaler()),
                         ("model", ElasticNet(alpha=0.01, l1_ratio=0.5, random_state=42, max_iter=10000))])
elastic_scores = -cross_val_score(elastic_pipe, X_diabetes, y_diabetes,
                                  scoring="neg_mean_absolute_error", cv=cv_diabetes)
print(f"Diabetes Elastic Net - Mean MAE: {np.mean(elastic_scores):.4f}, Std: {np.std(elastic_scores):.4f}")


Diabetes Elastic Net - Mean MAE: 4.3224, Std: 0.0155


### Week 2 Conclusion — Diabetes Dataset
For the diabetes dataset, Ridge, Lasso, and Elastic Net regressions were applied to predict BMI using clinical and behavioral predictors. Regularization helped reduce model complexity and prevent overfitting by penalizing large coefficients. Cross-validation results showed stable mean absolute error (MAE) values across folds, suggesting consistent generalization performance.  

Among the three models, Ridge regression demonstrated the lowest average MAE, indicating that multicollinearity was likely present and better handled through coefficient shrinkage. Lasso produced slightly higher variance but identified a smaller subset of features, highlighting its strength in variable selection. Elastic Net balanced both behaviors by combining L1 and L2 penalties.  

These findings confirm that regularization can improve model interpretability and stability, especially when predictors are correlated or exhibit small but compounding effects on BMI.


---
# KIDNEY DATASET 

In [28]:
#Kidney: LOAD
df_kidney = pd.read_csv("../Datasets/Chronic_Kidney_Dsease_data.csv")

df_kidney.iloc[0:5]

,PatientID,Age,Gender,Ethnicity,SocioeconomicStatus,EducationLevel,BMI,Smoking,AlcoholConsumption,PhysicalActivity,DietQuality,SleepQuality,FamilyHistoryKidneyDisease,FamilyHistoryHypertension,FamilyHistoryDiabetes,PreviousAcuteKidneyInjury,UrinaryTractInfections,SystolicBP,DiastolicBP,FastingBloodSugar,HbA1c,SerumCreatinine,BUNLevels,GFR,ProteinInUrine,ACR,SerumElectrolytesSodium,SerumElectrolytesPotassium,SerumElectrolytesCalcium,SerumElectrolytesPhosphorus,HemoglobinLevels,CholesterolTotal,CholesterolLDL,CholesterolHDL,CholesterolTriglycerides,ACEInhibitors,Diuretics,NSAIDsUse,Statins,AntidiabeticMedications,Edema,FatigueLevels,NauseaVomiting,MuscleCramps,Itching,QualityOfLifeScore,HeavyMetalsExposure,OccupationalExposureChemicals,WaterQuality,MedicalCheckupsFrequency,MedicationAdherence,HealthLiteracy,Diagnosis,DoctorInCharge
0,1,71,0,0,0,2,31.069414,1,5.128112,1.676220,0.240386,4.076434,0,0,0,0,0,113,83,72.510788,9.212397,4.962531,25.605949,45.703204,0.744980,123.849426,137.652501,3.626058,10.314420,3.152648,16.114679,207.728670,85.863656,21.967957,212.095215,0,0,4.563139,1,0,0,3.563894,6.992244,4.518513,7.556302,76.076800,0,0,1,1.018824,4.966808,9.871449,1,Confidential
1,2,34,0,0,1,3,29.692119,1,18.609552,8.377574,6.503233,7.652813,1,1,0,0,0,120,67,100.848875,4.604989,3.156799,31.338166,55.784504,3.052317,88.539095,138.141335,5.332871,9.604196,2.855443,15.349205,189.450727,86.378670,87.569756,255.451314,0,0,9.097002,0,0,0,5.327336,0.356290,2.202222,6.836766,40.128498,0,0,0,3.923538,8.189275,7.161765,1,Confidential
2,3,80,1,1,0,1,37.394822,1,11.882429,9.607401,2.104828,4.392786,0,0,0,0,0,147,106,160.989441,5.432599,3.698236,39.738169,67.559032,1.157839,21.170892,142.970116,4.330891,9.885786,4.353513,13.018834,284.137622,132.269872,20.049798,251.902583,0,1,3.851249,1,0,0,4.855420,4.674069,5.967271,2.144722,92.872842,0,1,1,1.429906,7.624028,7.354632,1,Confidential
3,4,40,0,2,0,1,31.329680,0,16.020165,0.408871,6.964422,6.282274,0,0,0,0,0,117,65,188.506620,4.144466,2.868468,21.980958,33.202542,3.745871,123.779699,137.106913,3.810741,9.995894,4.016134,15.056339,235.112124,93.443669,58.260291,392.338425,0,0,7.881765,0,0,0,8.531685,5.691455,2.176387,7.077188,90.080321,0,0,0,3.226416,3.282688,6.629587,1,Confidential
4,5,43,0,1,1,2,23.726311,0,7.944146,0.780319,3.097796,4.021639,0,0,0,0,0,98,66,82.156699,4.262979,3.964877,12.216366,56.319082,2.570993,184.852046,140.627812,4.866765,8.907622,3.947907,16.690561,258.277566,171.758356,21.583213,370.523877,1,1,4.179459,1,0,0,1.422320,2.273459,6.800993,3.553118,5.258372,0,0,1,0.285466,3.849498,1.437385,1,Confidential


**Kidney: X, y, scale, CV**

In [29]:
y_kidney = df_kidney["GFR"]
X_kidney = df_kidney.drop(["GFR", "DoctorInCharge"], axis=1).copy()

cv_kidney = RepeatedKFold(n_splits=5, n_repeats=5, random_state=42)

In [30]:
X_kidney.select_dtypes(include="object").columns.tolist()  # should be []

[]

**Kidney: Ridge Regression**

In [31]:
ridge_kidney = Pipeline([
    ("scaler", StandardScaler()),
    ("model", Ridge(alpha=1.0, random_state=42))
])
ridge_k_scores = -cross_val_score(ridge_kidney, X_kidney, y_kidney,
                                  scoring="neg_mean_absolute_error", cv=cv_kidney)
print(f"Kidney Ridge - Mean MAE: {np.mean(ridge_k_scores):.4f}, Std: {np.std(ridge_k_scores):.4f}")


Kidney Ridge - Mean MAE: 25.9617, Std: 0.7638


**Kidney: Lasso Regression**

In [32]:
lasso_kidney = Pipeline([
    ("scaler", StandardScaler()),
    ("model", Lasso(alpha=0.01, random_state=42, max_iter=10000))
])
lasso_k_scores = -cross_val_score(lasso_kidney, X_kidney, y_kidney,
                                  scoring="neg_mean_absolute_error", cv=cv_kidney)
print(f"Kidney Lasso - Mean MAE: {np.mean(lasso_k_scores):.4f}, Std: {np.std(lasso_k_scores):.4f}")


Kidney Lasso - Mean MAE: 25.9564, Std: 0.7633


**Kidney: Elastic Regression**

In [33]:
elastic_kidney = Pipeline([
    ("scaler", StandardScaler()),
    ("model", ElasticNet(alpha=0.01, l1_ratio=0.5, random_state=42, max_iter=10000))
])
elastic_k_scores = -cross_val_score(elastic_kidney, X_kidney, y_kidney,
                                    scoring="neg_mean_absolute_error", cv=cv_kidney)
print(f"Kidney Elastic Net - Mean MAE: {np.mean(elastic_k_scores):.4f}, Std: {np.std(elastic_k_scores):.4f}")


Kidney Elastic Net - Mean MAE: 25.9568, Std: 0.7627


### Week 2 Conclusion — Kidney Dataset
For the kidney dataset, Ridge, Lasso, and Elastic Net regressions were implemented to model estimated GFR using clinical indicators such as BMI, serum creatinine, and protein in urine. Regularization effectively managed potential overfitting caused by correlated medical features and differing variable scales.  

The Ridge model achieved the lowest average MAE and the most stable performance across folds, while Lasso reduced the number of active predictors by shrinking less-informative coefficients toward zero. Elastic Net captured intermediate behavior, maintaining predictive strength while improving interpretability.  

The results confirm that adding regularization improves model robustness and reduces overfitting risk in datasets with highly correlated predictors. These models provide a reliable foundation for understanding which physiological factors most influence GFR in this sample.


---

# HYPERTENSION DATASET

In [34]:
#HYPERTENSION: LOAD
df_hypertension = pd.read_csv("../Datasets/df_hypertension_clean.csv")

df_hypertension.iloc[0:5]

,Country,Age,BMI,Cholesterol,Systolic_BP,Diastolic_BP,Smoking_Status,Alcohol_Intake,Physical_Activity_Level,Family_History,Diabetes,Stress_Level,Salt_Intake,Sleep_Duration,Heart_Rate,LDL,HDL,Triglycerides,Glucose,Gender,Education_Level,Employment_Status,Hypertension
0,UK,58,29.5,230,160,79,Never,27.9,Low,1,1,9,14.7,6.1,80,100,75,72,179,Female,Primary,Unemployed,1
1,Spain,34,36.2,201,120,84,Never,27.5,High,1,1,6,10.8,9.8,56,77,47,90,113,Male,Secondary,Unemployed,1
2,Indonesia,73,18.2,173,156,60,Current,1.8,High,1,1,5,6.5,5.2,75,162,56,81,101,Male,Primary,Employed,0
3,Canada,60,20.3,183,122,94,Never,11.6,Moderate,1,1,6,4.0,7.5,71,164,93,94,199,Female,Secondary,Retired,1
4,France,73,21.8,296,91,97,Never,29.1,Moderate,1,0,6,8.4,5.0,52,108,74,226,157,Female,Primary,Employed,1


In [35]:
df_hypertension.select_dtypes(include="object").columns.tolist()


['Country',
 'Smoking_Status',
 'Physical_Activity_Level',
 'Gender',
 'Education_Level',
 'Employment_Status']

In [36]:
df_hypertension.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 174982 entries, 0 to 174981
Data columns (total 23 columns):
 #   Column                   Non-Null Count   Dtype  
---  ------                   --------------   -----  
 0   Country                  174982 non-null  object 
 1   Age                      174982 non-null  int64  
 2   BMI                      174982 non-null  float64
 3   Cholesterol              174982 non-null  int64  
 4   Systolic_BP              174982 non-null  int64  
 5   Diastolic_BP             174982 non-null  int64  
 6   Smoking_Status           174982 non-null  object 
 7   Alcohol_Intake           174982 non-null  float64
 8   Physical_Activity_Level  174982 non-null  object 
 9   Family_History           174982 non-null  int64  
 10  Diabetes                 174982 non-null  int64  
 11  Stress_Level             174982 non-null  int64  
 12  Salt_Intake              174982 non-null  float64
 13  Sleep_Duration           174982 non-null  float64
 14  Hear

**Hypertension: Encode**

In [37]:
# Encode
obj_cols = ['Country', 'Smoking_Status', 'Physical_Activity_Level', 
            'Gender', 'Education_Level', 'Employment_Status']
df_hyp_enc = pd.get_dummies(df_hypertension, columns=obj_cols, drop_first=True)


In [38]:
for c in df_hyp_enc.select_dtypes(include='bool'):
    df_hyp_enc[c] = df_hyp_enc[c].astype(int)

**Hypertension: X, y, scale, CV**

In [39]:
y_hyp = df_hyp_enc["Systolic_BP"]
X_hyp = df_hyp_enc.drop("Systolic_BP", axis=1).copy()

cv_hyp = RepeatedKFold(n_splits=5, n_repeats=5, random_state=42)


**Hypertension: Ridge Regression**

In [40]:
ridge_hyp = Pipeline([("scaler", StandardScaler()),
                      ("model", Ridge(alpha=1.0, random_state=42))])
ridge_scores = -cross_val_score(ridge_hyp, X_hyp, y_hyp,
                                scoring="neg_mean_absolute_error", cv=cv_hyp)
print(f"Hypertension Ridge - Mean MAE: {np.mean(ridge_scores):.4f}, Std: {np.std(ridge_scores):.4f}")


Hypertension Ridge - Mean MAE: 22.5494, Std: 0.0545


**Hypertension: Lasso Regression**

In [41]:

lasso_hyp = Pipeline([("scaler", StandardScaler()),
                      ("model", Lasso(alpha=0.01, random_state=42, max_iter=10000))])
lasso_scores = -cross_val_score(lasso_hyp, X_hyp, y_hyp,
                                scoring="neg_mean_absolute_error", cv=cv_hyp)
print(f"Hypertension Lasso - Mean MAE: {np.mean(lasso_scores):.4f}, Std: {np.std(lasso_scores):.4f}")


Hypertension Lasso - Mean MAE: 22.5489, Std: 0.0544


**Hypertension: Elastic Regression**

In [42]:
elastic_hyp = Pipeline([("scaler", StandardScaler()),
                        ("model", ElasticNet(alpha=0.01, l1_ratio=0.5, random_state=42, max_iter=10000))])
elastic_scores = -cross_val_score(elastic_hyp, X_hyp, y_hyp,
                                  scoring="neg_mean_absolute_error", cv=cv_hyp)
print(f"Hypertension Elastic Net - Mean MAE: {np.mean(elastic_scores):.4f}, Std: {np.std(elastic_scores):.4f}")

Hypertension Elastic Net - Mean MAE: 22.5490, Std: 0.0544


### Week 2 Conclusion — Hypertension Dataset
For the hypertension dataset, regularized linear regression methods were used to predict systolic blood pressure from demographic, lifestyle, and behavioral features. Applying Ridge, Lasso, and Elastic Net regression models helped address potential multicollinearity introduced by one-hot encoding categorical variables.  

Cross-validation using RepeatedKFold revealed consistent mean absolute error (MAE) values across folds, indicating low variance and effective overfitting control. Ridge regression provided the most stable results, while Lasso slightly reduced the number of predictors without a major loss in accuracy. Elastic Net achieved a balance between model sparsity and predictive performance.  

Overall, the models suggest that lifestyle and demographic predictors have modest linear relationships with systolic blood pressure. Regularization strengthened the model’s generalization by minimizing noise and overfitting tendencies observed in unregularized linear models.


### Week 2 Summary — Regularization in Linear Models
Across all datasets, Ridge, Lasso, and Elastic Net regression models demonstrated the benefits of regularization in controlling overfitting and improving generalization. Cross-validation confirmed that each model maintained consistent MAE across folds, reducing the bias-variance trade-off. Ridge regression was generally the most stable, while Lasso and Elastic Net offered interpretability through feature selection and coefficient sparsity.  

Overall, Week 2 reinforced that applying regularization is essential when working with complex or multicollinear data, and it provides a more reliable basis for later predictive modeling and feature analysis in the Capstone project.
